<!-- SPDX-FileCopyrightText: 2026 Orbital Research Cluster for Celestial Applications (ORCCA) Lab, University of Colorado at Boulder -->
<!-- SPDX-License-Identifier: ISC -->
# Trajectory
---
Last revised by Z. Ellis on 2026 APR 6

## Objectives
This tutorial will demonstrate 

## See Also
- Related Tutorial

## Imports and Set Up

Here we'll import the necessary libraries and load in the tutorials data folder. Then we define units and frames and load a metakernel for SPICE functionalities.

In [1]:
import scarabaeus as scb
import supplementary as supp

import os
import numpy as np
import matplotlib.pyplot as plt

# load tutorial data
data = supp.load_data()

SCB tutorial data up to date.


## Content of Tutorial
Info

In [2]:
import scarabaeus as scb
import supplementary as supp

import os
import numpy as np
import matplotlib.pyplot as plt

# load tutorial data
data = supp.load_data()

kg, km, sec = scb.Units.get_units(['kg', 'km', 'sec'])
J2000 = scb.Frame('J2000')
scb.SpiceManager.load_kernel_from_mkfile(data.mk.path)

## earth and spacecraft
earth   = scb.CelestialBody.from_constants("EARTH")
sc      = scb.Spacecraft(name      = 'ORCCA_SC',
                         spice_id  = -1000,
                         tot_mass  = scb.ArrayWUnits(2000, kg),
                         area      = scb.ArrayWUnits(1e-06, km**2),
                         ref_coeff = scb.ArrayWUnits(1.5, None))

## orbital parameters
t0     = scb.SpiceManager.cal2et('2026 JAN 01 00:00:00.000')
tf     = scb.SpiceManager.cal2et('2026 JAN 01 10:00:00.000')    # 10 hours
epochs = scb.EpochArray(np.arange(t0, tf, 500), 'TDB')

# semi major axis, ecccentricity, inclination, gravitational parameter
a, e, i, mu, nu = 25000, 0.5, np.deg2rad(35), earth.grav_param.values, 0

# perifocal position and velocity
h  = np.sqrt(mu*a*(1 - e**2))        # angular momentum 
r  = h**2 / (mu*(1 + e*np.cos(nu)))
r0 = np.array([r*np.cos(nu), r*np.sin(nu), 0])
v0 = np.array([-(mu/h)*np.sin(nu), (mu/h) * (e + np.cos(nu)), 0])

# rotate to intertial intertial
peri_to_n = np.array([[1,         0,          0],
                        [0, np.cos(i), -np.sin(i)],
                        [0, np.sin(i), np.cos(i)]])
r0 = (peri_to_n @ r0.reshape((3, 1))).flatten()
v0 = (peri_to_n @ v0.reshape((3, 1))).flatten()

# convert to AWF and return
pos0 = scb.ArrayWFrame(r0, km    , J2000)
vel0 = scb.ArrayWFrame(v0, km/sec, J2000)

state_vector = scb.StateArray(epoch  = epochs[0],
                                origin = earth,
                                state  = scb.StateDefinition()
                                            .position(sc, pos0)
                                            .velocity(sc, vel0))

twobp = scb.ForceModelTranslation(primary_body = sc)
prop  = scb.Propagator(primary_body  = sc,
                                   state_vector  = state_vector,
                                   tspan         = epochs,
                                   integrator    = 'IAS15',
                                   force_models  = twobp,
                                   propagate_STM = False,
                                   display_progress= False)
prop.propagate()
ts_ias = prop.times.times.values
ys_ias = prop.ys


# single state array
test_traj = scb.Trajectory(state_array = prop.propagated_state_array)
if os.path.exists('/Users/zael5647/scarabaeus/tutorials/TEST_STATEARRAY.bsp'):
    os.remove('/Users/zael5647/scarabaeus/tutorials/TEST_STATEARRAY.bsp')
test_traj.write_to_spk('TEST_STATEARRAY.bsp')
scb.SpiceManager.unload_kernel_from_pool('/Users/zael5647/scarabaeus/tutorials/TEST_STATEARRAY.bsp')
scb.SpiceManager.load_kernel('/Users/zael5647/scarabaeus/tutorials/TEST_STATEARRAY.bsp')
x0_from_spk = scb.SpiceManager.get_state(sc, t0, J2000, earth)
# print(x0_from_spk)
# print(pos0, vel0)






# # ### mission sequence
# sun = scb.CelestialBody.from_constants('SUN')
# time_0 = scb.SpiceManager.jd2et(2461809.72995654 + 1 / 3)
# time_f = scb.SpiceManager.jd2et(2461809.72995654 + 4)
# dt = dt = 30 * 30 * 30
# epoch_array = scb.EpochArray(np.arange(time_0, time_f, dt), sys="TDB")

# sc = scb.Spacecraft(
#         "TEST_SC",
#         -1099,
#         scb.ArrayWUnits(1275.0, kg),
#         scb.ArrayWUnits(1e-06, km**2),
#         scb.ArrayWUnits(1, None),
#     )

# # first leg
# leg1_name = "First Leg"
# leg1_period = scb.ArrayWUnits(58000, sec)
# pos1 = scb.ArrayWFrame(np.array([7000.0, 0.0, 0.0]), km, J2000)
# vel1 = scb.ArrayWFrame(np.array([0.0, 7.546049, 0.0]), km / sec, J2000)

# leg1_model = scb.StateArray(epoch  = epoch_array[0],
#                             origin = sun,
#                             state  = scb.StateDefinition()
#                                         .position(sc, pos1)
#                                         .velocity(sc, vel1))
# leg1_dt = scb.ArrayWUnits(dt, sec)

# # first node
# dv1_name = "Impulsive Burn"
# dv1_model = scb.ImpulsiveBurn(scb.ArrayWFrame(scb.ArrayWUnits(np.array([5]), km / sec), J2000),
#                               scb.ArrayWFrame(scb.ArrayWUnits(np.array([0.16384638, 0.9830783, 0.08192319]), km / km), J2000))

# # second leg
# leg2_name = "Second Leg"
# leg2_period = scb.ArrayWUnits(58000, sec)
# pos2 = scb.ArrayWFrame(np.full(3, np.nan), km, J2000)
# vel2 = scb.ArrayWFrame(np.full(3, np.nan), km / sec, J2000)

# leg2_model = scb.StateArray(epoch  = scb.EpochArray(np.full(1, np.nan), "TDB"),
#                             origin = sun,
#                             state  = scb.StateDefinition()
#                                         .position(sc, pos2)
#                                         .velocity(sc, vel2))
# leg2_dt = scb.ArrayWUnits(dt, sec)


# # Define propagator
# third_bodies = ["MERCURY", "VENUS", "EARTH"]
# force_model = scb.ForceModelTranslation(
#     primary_body=sc, cannonball_SRP=True, third_bodies=third_bodies
# )
# prop_model1 = scb.Propagator(
#     primary_body=sc,
#     state_vector=leg1_model,
#     tspan=epoch_array,
#     force_models=force_model,
# )

# prop_model1.propagate()

# prop_model2 = scb.Propagator(
#     primary_body=sc,
#     state_vector=leg2_model,
#     tspan=epoch_array,
#     force_models=force_model,
# )

# # =======================================================
# # Mission sequence definition and results
# # =======================================================

# # Create mission sequence
# MS = scb.MissionSequence("ORCCA_SEQUENCE")
# MS.addLeg(leg1_name, leg1_period, leg1_model, prop_model1, leg1_dt)
# # MS.addBurn(dv1_name, dv1_model, prop_model1)
# # MS.addLeg(leg2_name, leg2_period, leg2_model, prop_model2, leg2_dt)

# scenario = MS.propagate()

# ref_sequence, ref_idx = False, 0
# name_file = f"orbiter_scenario_desat{'_ref' if ref_sequence else ''}{f'{ref_idx}' if ref_idx != 0 else ''}.bsp"
# print(name_file)
# # orbiter_traj = scb.Trajectory(
# #     name=name_file,
# #     leg_array=scenario,
# # )
# # orbiter_traj.add_STMs(scenario.STM)

SCB tutorial data up to date.


SpiceNOSUCHFILE: 
================================================================================

Toolkit version: CSPICE_N0067

SPICE(NOSUCHFILE) --

The attempt to load "/Users/zael5647/scarabaeus/tutorials/TEST_STATEARRAY.bsp" by the routine FURNSH failed. It could not be located.

furnsh_c --> FURNSH --> ZZLDKER

================================================================================

In [3]:
import scarabaeus as scb
import supplementary as supp

import os
import numpy as np
import matplotlib.pyplot as plt

# load tutorial data
data = supp.load_data()

kg, km, sec = scb.Units.get_units(['kg', 'km', 'sec'])
J2000 = scb.Frame('J2000')
scb.SpiceManager.load_kernel_from_mkfile(data.mk.path)

Orbiter_drymass = scb.ArrayWUnits(1275.0, kg)
Orbiter_fuelmass = scb.ArrayWUnits(135, kg)
Orbiter_area = scb.ArrayWUnits(1e-06, km**2)
Orbiter_cr_srp = scb.ArrayWUnits(1, None)
sc_true = scb.Spacecraft(
    "TEST_SC",
    -1099,
    Orbiter_drymass + Orbiter_fuelmass,
    Orbiter_area,
    Orbiter_cr_srp,
)
spacecraft = sc_true

# Initialize frame, planet constants and origin
frame = J2000
origin = scb.CelestialBody.from_constants("SUN")

# Initialize epochs
time_0 = scb.SpiceManager.jd2et(2461809.72995654 + 1 / 3)
time_f = scb.SpiceManager.jd2et(2461809.72995654 + 4)
period = scb.ArrayWUnits(time_f - time_0, sec)
dt = 30 * 30 * 30
epoch_array = scb.EpochArray(np.arange(time_0, time_f, dt), sys="TDB")

# Get initial position and velocity
initial_position = scb.ArrayWUnits(
    np.array([-1.1123095885148e08, 8.9094345479316e07, 3.8656500948069e07]), km
)
initial_velocity = scb.ArrayWUnits(
    np.array([-20.6936999825159, -16.7800270812616, -6.6437327193572]), km / sec
)

# Apply perturbation if the flag is set
initial_position = scb.ArrayWFrame(initial_position, frame)
initial_velocity = scb.ArrayWFrame(initial_velocity, frame)

# Initialize first leg
leg1_name = "First Leg"
leg1_period = period
leg1_state_definition = [
    (
        "position",
        3,
        "estimated",
        "dynamic",
        spacecraft,
        initial_position,
    ),
    (
        "velocity",
        3,
        "estimated",
        "dynamic",
        spacecraft,
        initial_velocity,
    ),
]
leg1_model = scb.StateArray(
    epoch=epoch_array[0],
    origin=origin,
    state=scb.StateDefinition.from_components(leg1_state_definition),
)
leg1_dt = scb.ArrayWUnits(dt, sec)

# Initialize first node
dv1_name = "Impulsive Burn"
dv1 = scb.ArrayWUnits(0 * np.array([0.16384638, 0.9830783, 0.08192319]), km / sec)
dv1 = scb.ArrayWFrame(dv1, frame)
dv1_model = scb.ImpulsiveBurn(delta_v=dv1)

# Initialize second leg
leg2_name = "Second Leg"
leg2_period = period
leg2_state_definition = [
    (
        "position",
        3,
        "estimated",
        "dynamic",
        spacecraft,
        scb.ArrayWFrame(np.zeros(3), km, frame),  # Dummy value
    ),
    (
        "velocity",
        3,
        "estimated",
        "dynamic",
        spacecraft,
        scb.ArrayWFrame(np.zeros(3), km / sec, frame),  # Dummy value
    ),
    (
        "dv_man",
        3,
        "estimated",
        "static",
        spacecraft,
        dv1,  # Perturbed value if flag is set
    ),
]
leg2_model = scb.StateArray(
    epoch=scb.EpochArray(np.zeros(1), sys="TDB"),
    origin=origin,
    state=scb.StateDefinition.from_components(leg2_state_definition),
)
leg2_dt = scb.ArrayWUnits(dt, sec)

# Define propagator
third_bodies = ["MERCURY", "VENUS", "EARTH"]
force_model = scb.ForceModelTranslation(
    primary_body=spacecraft, third_bodies=third_bodies, cannonball_SRP=True
)
prop_model1 = scb.Propagator(
    primary_body=spacecraft,
    state_vector=leg1_model,
    tspan=epoch_array,
    force_models=force_model,
)

prop_model2 = scb.Propagator(
    primary_body=spacecraft,
    state_vector=leg2_model,
    tspan=epoch_array,
    force_models=force_model,
)

# Create mission sequence
MS = scb.MissionSequence("ORCCA_sequence_desatOD")
MS.addLeg(leg1_name, leg1_period, leg1_model, prop_model1, leg1_dt)
MS.addBurn(dv1_name, dv1_model, prop_model1)
MS.addLeg(leg2_name, leg2_period, leg2_model, prop_model2, leg2_dt)

scenario = MS.propagate()

###### here
orbiter_traj = scb.Trajectory(leg_array = scenario)
orbiter_traj.add_STMs(scenario.STM)


# if os.path.exists('/Users/zael5647/scarabaeus/tutorials/TEST_LEG.bsp'):
#     os.remove('/Users/zael5647/scarabaeus/tutorials/TEST_LEG.bsp')
orbiter_traj.write_to_spk('TEST_LEG.bsp')
scb.SpiceManager.unload_kernel_from_pool('/Users/zael5647/scarabaeus/tutorials/TEST_LEG.bsp')
scb.SpiceManager.load_kernel('/Users/zael5647/scarabaeus/tutorials/TEST_LEG.bsp')
x0_from_spk = scb.SpiceManager.get_state(spacecraft, epoch_array[0], J2000, origin)
print(x0_from_spk)
print(initial_position, initial_velocity)

SCB tutorial data up to date.

--- Propagating Segment 1/3: 'First Leg' [Leg] ---

[IC] segment='First Leg' type='Leg' epoch=886901537.4300113 sec (TDB)
  position     n= 3 sid=-1099 frame=J2000 vals[:6]=[-1.11230959e+08  8.90943455e+07  3.86565009e+07]
  velocity     n= 3 sid=-1099 frame=J2000 vals[:6]=[-20.69369998 -16.78002708  -6.64373272]

                            Starting propagation...                             


/Users/zael5647/scarabaeus/src/scarabaeus/dynamics/ImpulsiveBurn.py:171: RuntimeWarning: invalid value encountered in divide
  self._dv_unit_vector = delta_v_val / self.dv_magnitude


Integrating:   0%|                                                      | 0.00/316800.00 s [00:00<?]

Integrating:   0%|                                                      | 0.00/316800.00 s [00:00<?]

Integrating:   0%|                                              | 0.01/316800.00 s [00:00<164:04:57]

...

Integrating: 100%|█████████████████████████████████████████████| 316800.00/316800.00 s [00:00<00:00]

Integrating: 100%|█████████████████████████████████████████████| 316800.00/316800.00 s [00:00<00:00]

Integrating: 100%|█████████████████████████████████████████████| 316800.00/316800.00 s [00:00<00:00]


 =================== DOP853 integration complete. ==================
Propagation complete.

[GLOBAL STATE after 'Second Leg']
  key=('position', 3, -1099, 'J2000', 0) -> global[0:3] = [-1.23368028e+08  7.75036083e+07  3.39707023e+07]
  key=('velocity', 3, -1099, 'J2000', 0) -> global[3:6] = [-17.62825578 -19.38850448  -7.88422558]
  key=('dv_man', 3, -1099, 'J2000', 0) -> global[6:9] = [0. 0. 0.]


SpiceNOSUCHFILE: 
================================================================================

Toolkit version: CSPICE_N0067

SPICE(NOSUCHFILE) --

The attempt to load "/Users/zael5647/scarabaeus/tutorials/TEST_LEG.bsp" by the routine FURNSH failed. It could not be located.

furnsh_c --> FURNSH --> ZZLDKER

================================================================================